In [8]:
import pandas as pd

alvr_og = pd.read_excel("2020_votereg_data_AlSoS.xlsx", sheet_name="October", header=None, engine="openpyxl")
rdh_og = pd.read_csv("AL_l2_2024_gen_stats_2020county.csv")
demographics_og = pd.read_csv("demographics_2020_general.csv")

alvr = alvr_og.copy()
rdh = rdh_og.copy()
demographics = demographics_og.copy()

rdh = rdh.rename(columns={"countyname": "county_name", "countyfips": "fips_code"})
rdh["county_name"] = rdh["county_name"].str.strip().str.title()

alvr.columns = [
    'county_name', 'Total_AI',
    'Active_Asian', 'Active_AmInd', 'Active_Black', 'Active_Fed',
    'Active_Hispanic', 'Active_Korean', 'Active_White', 'Active_Other',
    'Active_NotId', 'Active_Total',
    'Inactive_Asian', 'Inactive_AmInd', 'Inactive_Black', 'Inactive_Fed',
    'Inactive_Hispanic', 'Inactive_Korean', 'Inactive_White', 'Inactive_Other',
    'Inactive_NotId', 'Inactive_Total'
]

alvr = alvr.iloc[2:].reset_index(drop=True)
alvr["county_name"] = alvr["county_name"].astype(str).str.strip().str.title()

alvr_numeric_columns = [
    'Active_Asian', 'Active_AmInd', 'Active_Black', 'Active_Fed', 'Active_Hispanic',
    'Active_Korean', 'Active_White', 'Active_Other', 'Active_NotId', 'Active_Total',
    'Inactive_Asian', 'Inactive_AmInd', 'Inactive_Black', 'Inactive_Fed', 'Inactive_Hispanic',
    'Inactive_Korean', 'Inactive_White', 'Inactive_Other', 'Inactive_NotId', 'Inactive_Total'
]
alvr[alvr_numeric_columns] = alvr[alvr_numeric_columns].fillna(0)

BLACK_BELT_COUNTIES = [
    "Sumter", "Choctaw", "Greene", "Hale", "Marengo", "Perry", "Dallas",
    "Wilcox", "Lowndes", "Butler", "Crenshaw", "Montgomery", "Pike",
    "Bullock", "Macon", "Barbour", "Russell"
]

alvr_bb = alvr[alvr["county_name"].isin(BLACK_BELT_COUNTIES)].copy()
rdh_bb = rdh[rdh["county_name"].isin(BLACK_BELT_COUNTIES)].copy()

registration = alvr_bb.merge(rdh_bb[["county_name", "fips_code"]], on="county_name")
registration = registration.merge(demographics[["county_name", "eligible_black", "eligible_white", "eligible_latinx"]], on="county_name")

registration["total_registered_black"] = registration["Active_Black"].astype(float) + registration["Inactive_Black"].astype(float)
registration["total_registered_white"] = registration["Active_White"].astype(float) + registration["Inactive_White"].astype(float)
registration["total_registered_latinx"] = registration["Active_Hispanic"].astype(float) + registration["Inactive_Hispanic"].astype(float)

registration["reg_active_rate_black"] = (registration["Active_Black"].astype(float) / registration["eligible_black"]).round(4)
registration["reg_active_rate_white"] = (registration["Active_White"].astype(float) / registration["eligible_white"]).round(4)
registration["reg_active_rate_latinx"] = (registration["Active_Hispanic"].astype(float) / registration["eligible_latinx"]).round(4)

registration["reg_total_rate_black"] = (registration["total_registered_black"] / registration["eligible_black"]).round(4)
registration["reg_total_rate_white"] = (registration["total_registered_white"] / registration["eligible_white"]).round(4)
registration["reg_total_rate_latinx"] = (registration["total_registered_latinx"] / registration["eligible_latinx"]).round(4)

registration["inactive_rate"] = (registration["Inactive_Total"].astype(float) / (registration["Active_Total"].astype(float) + registration["Inactive_Total"].astype(float))).round(4)
registration["inactive_rate_black"] = (registration["Inactive_Black"].astype(float) / registration["total_registered_black"]).round(4)
registration["inactive_rate_white"] = (registration["Inactive_White"].astype(float) / registration["total_registered_white"]).round(4)
registration["inactive_rate_latinx"] = (registration["Inactive_Hispanic"].astype(float) / registration["total_registered_latinx"]).round(4)

registration = registration.rename(columns={
    "Active_Black": "active_black",
    "Active_White": "active_white",
    "Active_Hispanic": "active_latinx",
    "Active_Total": "active_total",
    "Inactive_Black": "inactive_black",
    "Inactive_White": "inactive_white",
    "Inactive_Hispanic": "inactive_latinx",
    "Inactive_Total": "inactive_total"
})

final_columns = [
    "county_name", "fips_code",
    "active_total", "inactive_total", "inactive_rate",
    "eligible_black", "active_black", "inactive_black", "total_registered_black",
    "reg_active_rate_black", "reg_total_rate_black", "inactive_rate_black",
    "eligible_white", "active_white", "inactive_white", "total_registered_white",
    "reg_active_rate_white", "reg_total_rate_white", "inactive_rate_white",
    "eligible_latinx", "active_latinx", "inactive_latinx", "total_registered_latinx",
    "reg_active_rate_latinx", "reg_total_rate_latinx", "inactive_rate_latinx"
]

registration = registration[final_columns].sort_values("county_name").reset_index(drop=True)
registration = registration.fillna("null")

registration.to_csv("voter_registration_2020_general.csv", index=False)
print("Created: voter_registration_2020_general.csv")
registration

Created: voter_registration_2020_general.csv


C:\Users\hingz\AppData\Local\Temp\ipykernel_34692\2415867102.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  alvr[alvr_numeric_columns] = alvr[alvr_numeric_columns].fillna(0)


,county_name,fips_code,active_total,inactive_total,inactive_rate,eligible_black,active_black,inactive_black,total_registered_black,reg_active_rate_black,...,reg_active_rate_white,reg_total_rate_white,inactive_rate_white,eligible_latinx,active_latinx,inactive_latinx,total_registered_latinx,reg_active_rate_latinx,reg_total_rate_latinx,inactive_rate_latinx
0,Barbour,1005,16799,1021,0.0573,9245,7903,431,8334.0,0.8548,...,0.8982,0.9574,0.0619,405,98,9,107.0,0.2420,0.2642,0.0841
1,Bullock,1011,6959,479,0.0644,5525,5309,372,5681.0,0.9609,...,0.8000,0.8505,0.0594,250,42,2,44.0,0.1680,0.1760,0.0455
2,Butler,1013,13839,765,0.0524,6564,6038,412,6450.0,0.9199,...,0.9282,0.9687,0.0418,175,35,8,43.0,0.2000,0.2457,0.1860
3,Choctaw,1023,10320,690,0.0627,4259,4504,379,4883.0,1.0575,...,0.9912,1.0441,0.0506,55,12,0,12.0,0.2182,0.2182,0.0000
4,Crenshaw,1041,10073,534,0.0503,2564,2455,136,2591.0,0.9575,...,0.9741,1.0250,0.0497,125,36,0,36.0,0.2880,0.2880,0.0000
5,Dallas,1047,29897,1893,0.0595,19904,21084,1277,22361.0,1.0593,...,0.9871,1.0525,0.0622,80,37,13,50.0,0.4625,0.6250,0.2600
6,Greene,1063,6697,340,0.0483,5085,5490,296,5786.0,1.0796,...,0.8976,0.9323,0.0372,40,19,0,19.0,0.4750,0.4750,0.0000
7,Hale,1065,11848,671,0.0536,6664,6923,395,7318.0,1.0389,...,1.0330,1.0897,0.0520,10,20,3,23.0,2.0000,2.3000,0.1304
8,Lowndes,1085,9751,592,0.0572,5654,7235,454,7689.0,1.2796,...,1.2371,1.3025,0.0503,30,20,4,24.0,0.6667,0.8000,0.1667
9,Macon,1087,15497,2403,0.1342,12499,12816,2186,15002.0,1.0254,...,0.9742,1.0382,0.0616,60,44,13,57.0,0.7333,0.9500,0.2281
